# S004 — LSE Post-Earnings Announcement Drift (PEAD)
**Signal:** SUE = EPS surprise / rolling 8Q EPS volatility, forward-filled for 21 trading days  
**Universe:** LSE canonical ~1,347 tickers, PEAD-eligible: 140 tickers (≥4 events)  
**Portfolio:** Long-only Top-8 active SUE positions, daily rebalance (21d hold natural expiry)  
**Costs:** 10 bps commission, ~18x/yr turnover

In [ ]:
import sys, os, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

RUN_AT = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")

sys.path.insert(0, str(Path(r"c:\Personal\Business & Investments\Python codes")))
from signum import Chart
from signum.engine.dashboard import Dashboard
from signum.engine.statchart import StatChart

def _find_btest_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "AGENT_DSL_REFERENCE.md").exists():
            return p
    return Path(r"c:\Personal\Business & Investments\Python codes\btest")

BTEST_ROOT  = _find_btest_root()
os.chdir(BTEST_ROOT)

SIGNAL_ROOT = Path("research/generated/Dividend Growth/signals/004_lse_pead")
OUTPUTS     = SIGNAL_ROOT / "outputs"
DATA_DIR    = SIGNAL_ROOT / "data"
SHARED_DATA = Path("research/generated/Dividend Growth/shared_data")

equity  = pd.read_parquet(OUTPUTS / "equity.parquet")
returns = pd.read_parquet(OUTPUTS / "returns.parquet")
trades  = pd.read_parquet(OUTPUTS / "trades.parquet")
weights = pd.read_parquet(OUTPUTS / "weights.parquet")

raw_sum = json.loads((OUTPUTS / "summary.json").read_text())
summary = raw_sum.get("metrics", raw_sum)

for df_ in [equity, weights]:
    if hasattr(df_.index, "tz") and df_.index.tz is not None:
        df_.index = df_.index.tz_localize(None)

eq_col    = next((c for c in equity.columns  if any(k in c.lower() for k in ("nav","portfolio","equity","total"))), equity.columns[0])
strat_col = next((c for c in returns.columns if any(k in c.lower() for k in ("strategy","total","portfolio"))),     returns.columns[0])
eq        = equity[eq_col].dropna()
strat_ret = returns[strat_col].dropna()

print("✓  S004 outputs loaded")
print(f"   Date range : {eq.index[0].date()} → {eq.index[-1].date()}")
print(f"   Trades     : {len(trades):,}")
print(f"   Universe   : {weights.shape[1]} tickers")


✓  S004 outputs loaded
   Date range : 2015-01-02 → 2025-12-31
   Trades     : 16,792
   Universe   : 100 tickers


In [2]:
ann = 252
r    = strat_ret.dropna()
cagr = (eq.iloc[-1] / eq.iloc[0]) ** (ann / len(r)) - 1
dd   = (eq - eq.cummax()) / eq.cummax()

metrics = {
    "Total Return"    : f"{eq.iloc[-1]/eq.iloc[0]-1:.1%}",
    "CAGR"            : f"{cagr:.1%}",
    "Sharpe"          : f"{r.mean()/r.std()*np.sqrt(ann):.2f}",
    "Sortino"         : f"{r.mean()/r[r<0].std()*np.sqrt(ann):.2f}",
    "Max Drawdown"    : f"{dd.min():.1%}",
    "Calmar"          : f"{cagr/abs(dd.min()):.2f}",
    "Ann Volatility"  : f"{r.std()*np.sqrt(ann):.1%}",
    "Avg Daily Trades": f"{len(trades)/len(eq):.1f}",
}

mdf = pd.DataFrame.from_dict(metrics, orient="index", columns=["Value"])
display(mdf.style
    .set_caption("S004 — LSE PEAD (SUE)  |  Key Metrics")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","14px"),("font-weight","bold"),("text-align","left")]},
        {"selector": "th",      "props": [("text-align","left")]},
        {"selector": "td",      "props": [("text-align","right"),("font-family","monospace")]},
    ])
    .set_properties(**{"width": "140px"})
)


,Value
Total Return,79.6%
CAGR,5.5%
Sharpe,0.44
Sortino,0.46
Max Drawdown,-43.1%
Calmar,0.13
Ann Volatility,14.3%
Avg Daily Trades,6.0


In [3]:
nav_idx  = eq / eq.iloc[0] * 100
dd_pct   = (eq - eq.cummax()) / eq.cummax() * 100
r_sharpe = (strat_ret.rolling(63, min_periods=63).mean()
            / strat_ret.rolling(63, min_periods=63).std()
            * np.sqrt(252))

nav_df = pd.DataFrame({"time": nav_idx.index, "value": nav_idx.values})
dd_df  = pd.DataFrame({"time": dd_pct.index,  "value": dd_pct.values})
rs_df  = pd.DataFrame({"time": r_sharpe.index, "value": r_sharpe.values})

ann = 252; r = strat_ret.dropna()
cagr       = (eq.iloc[-1] / eq.iloc[0]) ** (ann / len(r)) - 1
sharpe_val = r.mean() / r.std() * np.sqrt(ann)

Dashboard(
    panes=[
        Chart(height=280).area(nav_df, name="NAV (rebased 100)", color="#26a69a"),
        Chart(height=130).area(dd_df,  name="Drawdown %",        color="#ef5350"),
        Chart(height=130).baseline(rs_df, base_value=0, value_col="value"),
    ],
    titles=[
        f"NAV  ·  CAGR {cagr:.1%}  ·  Total Return {eq.iloc[-1]/eq.iloc[0]-1:.1%}",
        f"Drawdown  ·  Max {dd_pct.min():.1f}%",
        f"Rolling Sharpe (63d)  ·  Full-period Sharpe {sharpe_val:.2f}",
    ],
    theme="dark",
)


In [4]:
monthly = strat_ret.resample("ME").apply(lambda x: (1+x).prod()-1)
pivot = monthly.rename_axis("date").to_frame("ret")
pivot["year"] = pivot.index.year; pivot["month"] = pivot.index.month
pivot = pivot.pivot(index="year", columns="month", values="ret") * 100
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot["Annual"] = strat_ret.resample("YE").apply(lambda x: (1+x).prod()-1) * 100

display(
    pivot.style
    .format("{:.1f}%", na_rep="")
    .background_gradient(cmap="RdYlGn", vmin=-8, vmax=8, subset=list(pivot.columns[:-1]))
    .background_gradient(cmap="RdYlGn", vmin=-20, vmax=20, subset=["Annual"])
    .set_caption("S004 — Monthly Returns (%)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center"),("min-width","48px")]},
        {"selector": "td",      "props": [("text-align","right"),("font-family","monospace"),("min-width","48px")]},
    ])
)

ann_ret = strat_ret.resample("YE").apply(lambda x: (1+x).prod()-1)
StatChart(theme="dark", height=220, title="Annual Returns Distribution").distribution(
    ann_ret * 100, bins=14, name="Annual Return %", color="#26a69a",
    show_mean=True, show_median=True,
).show()


,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Annual
year,,,,,,,,,,,,,
2015,0.0%,1.1%,-0.2%,0.0%,-1.0%,-0.3%,-0.1%,-3.6%,-2.8%,0.0%,1.8%,0.0%,
2016,0.0%,6.3%,-2.2%,-0.4%,0.1%,-0.8%,-0.1%,0.9%,0.8%,0.0%,-1.4%,-1.6%,
2017,0.0%,2.9%,1.4%,0.1%,5.6%,0.7%,-0.0%,2.9%,-0.2%,1.7%,-2.3%,0.1%,
2018,-0.0%,-0.9%,-2.3%,4.8%,2.9%,-0.9%,1.9%,-0.9%,-0.0%,-7.0%,3.3%,-5.0%,
2019,3.9%,-0.7%,-0.2%,3.6%,-0.5%,1.9%,2.1%,-1.4%,0.8%,-2.0%,1.1%,3.2%,
2020,1.2%,-10.1%,-22.8%,7.9%,4.0%,4.5%,1.6%,1.3%,-0.2%,-2.9%,12.5%,4.4%,
2021,1.5%,-0.8%,3.8%,4.3%,1.7%,4.6%,2.2%,3.3%,-0.2%,1.5%,1.8%,0.4%,
2022,0.0%,0.1%,-0.9%,-1.2%,0.5%,-8.2%,2.0%,-1.0%,0.4%,8.7%,8.8%,3.2%,
2023,0.4%,-2.5%,0.8%,-1.1%,-4.2%,1.5%,1.7%,-5.0%,-1.1%,1.7%,2.5%,1.6%,


In [5]:
latest_w = weights.iloc[-1].dropna()
latest_w = latest_w[latest_w > 0.001].sort_values(ascending=False) * 100

if len(latest_w):
    wdf = latest_w.reset_index()
    wdf.columns = ["Ticker", "Weight %"]
    wdf.index = range(1, len(wdf)+1)
    display(
        wdf.style
        .format({"Weight %": "{:.2f}%"})
        .bar(subset=["Weight %"], color="#26a69a", vmin=0)
        .set_caption(f"S004 — Active PEAD Positions at {weights.index[-1].date()}  ({len(wdf)} positions)")
    )
else:
    print("No active positions at latest date (all SUE windows expired)")

n_active_pos = (weights > 0.001).sum(axis=1).resample("ME").mean()
Chart(height=180, theme="dark", watermark="Avg Active Positions / Month").area(
    pd.DataFrame({"time": n_active_pos.index, "value": n_active_pos.values}),
    name="Active PEAD positions", color="#26a69a",
)


No active positions at latest date (all SUE windows expired)


In [6]:
sue = pd.read_parquet(DATA_DIR / "sue_signal.parquet")
if hasattr(sue.index, "tz") and sue.index.tz is not None:
    sue.index = sue.index.tz_localize(None)

n_active_sue = sue.notna().sum(axis=1).resample("ME").mean()
n_held       = (weights > 0.001).sum(axis=1).resample("ME").mean()

Dashboard(
    panes=[
        Chart(height=180).area(
            pd.DataFrame({"time": n_active_sue.index, "value": n_active_sue.values}),
            name="Active SUE signals", color="#1976d2"),
        Chart(height=130).area(
            pd.DataFrame({"time": n_held.index, "value": n_held.values}),
            name="Positions held", color="#ff9800"),
    ],
    titles=[
        f"Active SUE signals (within 21d window)  ·  avg {n_active_sue.mean():.1f}/month",
        f"Avg Positions Held  ·  avg {n_held.mean():.1f}",
    ],
    theme="dark",
)


In [7]:
events = pd.read_parquet(DATA_DIR / "events.parquet")

display(events.head(6).style.set_caption("events.parquet — sample rows"))
print(f"Total events: {len(events):,} | Tickers: {events['ticker'].nunique()}")

# SUE distribution
StatChart(theme="dark", height=260, title="SUE Distribution across all earnings events").distribution(
    events["sue"].dropna(), bins=60, name="SUE (sigma)",
    show_mean=True, show_median=True, fit=True,
    percentiles=[5, 25, 75, 95],
).show()

# Events per quarter
events["report_date"] = pd.to_datetime(events["report_date"])
qtr_counts = events.groupby(events["report_date"].dt.to_period("Q")).size().reset_index()
qtr_counts.columns = ["period", "count"]
qtr_counts["time"] = qtr_counts["period"].dt.to_timestamp()

Chart(height=200, theme="dark", watermark="Earnings Events per Quarter").histogram(
    qtr_counts[["time","count"]].rename(columns={"count":"value"}),
    name="Events",
)


,ticker,date,report_date,eps_actual,eps_estimate,eps_difference,before_after_market,sue
0,3IN,2018-03-31 00:00:00,2018-05-11 00:00:00,0.360000,0.000000,0.360000,None,5.000000
1,3IN,2018-09-30 00:00:00,2018-11-08 00:00:00,0.190000,0.000000,0.190000,None,1.682668
2,3IN,2019-03-31 00:00:00,2019-05-09 00:00:00,0.130000,0.000000,0.130000,None,1.277830
3,3IN,2019-09-30 00:00:00,2019-11-07 00:00:00,0.130000,0.000000,0.130000,None,1.385805
4,3IN,2020-03-31 00:00:00,2020-05-07 00:00:00,0.130000,0.000000,0.130000,None,1.485801
5,3IN,2022-09-30 00:00:00,2022-11-08 00:00:00,0.000000,0.000000,0.000000,BeforeMarket,0.000000


Total events: 2,282 | Tickers: 140


---

## Per-Ticker Attribution

Decompose portfolio returns by individual ticker using daily `weight × price return`. Shows which names drove performance, how long each was held, and trade-level realized P&L.

In [8]:
prices_long = pd.read_parquet(SHARED_DATA / "lse_prices.parquet")
prices_wide = prices_long.pivot_table(index="date", columns="ticker", values="close")
price_ret   = prices_wide.sort_index().pct_change(fill_method=None).clip(-0.5, 0.5)

w = weights.fillna(0.0)
w.index = pd.to_datetime(w.index).normalize()
common_tickers = w.columns.intersection(price_ret.columns)
common_dates   = w.index.intersection(price_ret.index)

w_sub = w.loc[common_dates, common_tickers]
r_sub = price_ret.loc[common_dates, common_tickers].fillna(0.0)

daily_contrib = w_sub.shift(1).fillna(0.0) * r_sub
ever_held     = (w_sub > 0.001).any()
contrib       = daily_contrib.loc[:, ever_held].sum()
n_days_held   = (w_sub.loc[:, ever_held] > 0.001).sum()
avg_wt        = w_sub.loc[:, ever_held].where(w_sub.loc[:, ever_held] > 0.001).mean()

attr = pd.DataFrame({
    "contrib_bps"    : (contrib * 10000).round(1),
    "contrib_pct"    : (contrib * 100).round(2),
    "days_held"      : n_days_held,
    "pct_time_held"  : (n_days_held / len(w_sub) * 100).round(1),
    "avg_weight_pct" : (avg_wt * 100).round(2),
}).sort_values("contrib_bps", ascending=False)

print(f"Tickers held: {ever_held.sum()} | Attribution sum: {contrib.sum()*100:.2f}% | Portfolio total: {(eq.iloc[-1]/eq.iloc[0]-1)*100:.2f}%")

display(attr.head(15).style
    .background_gradient(subset=["contrib_bps"], cmap="Greens")
    .format({"contrib_pct":"{:.2f}%","pct_time_held":"{:.1f}%","avg_weight_pct":"{:.2f}%"})
    .set_caption("Top 15 Contributors"))
display(attr.tail(10).style
    .background_gradient(subset=["contrib_bps"], cmap="Reds_r")
    .format({"contrib_pct":"{:.2f}%","pct_time_held":"{:.1f}%","avg_weight_pct":"{:.2f}%"})
    .set_caption("Bottom 10 Detractors"))


Tickers held: 0 | Attribution sum: 0.00% | Portfolio total: 79.65%


,contrib_bps,contrib_pct,days_held,pct_time_held,avg_weight_pct


,contrib_bps,contrib_pct,days_held,pct_time_held,avg_weight_pct


In [9]:
top10_tickers = attr.head(10).index.tolist()
bot5_tickers  = attr.tail(5).index.tolist()

cum_top  = (daily_contrib[top10_tickers].cumsum() * 100)
cum_bot  = (daily_contrib[bot5_tickers].cumsum() * 100)

top_plot = pd.DataFrame({"time": cum_top.index})
for tk in top10_tickers:
    top_plot[tk] = cum_top[tk].values

bot_plot = pd.DataFrame({"time": cum_bot.index})
for tk in bot5_tickers:
    bot_plot[tk] = cum_bot[tk].values

c_top = Chart(height=280, theme="dark", watermark="Cumulative Contribution — Top 10 Tickers (%)")
for tk in top10_tickers:
    c_top.line(top_plot[["time", tk]].rename(columns={tk: "value"}), name=tk)

c_bot = Chart(height=200, theme="dark", watermark="Cumulative Contribution — Bottom 5 Tickers (%)")
for tk in bot5_tickers:
    c_bot.line(bot_plot[["time", tk]].rename(columns={tk: "value"}), name=tk)

display(c_top)
display(c_bot)


In [10]:
scatter_df = attr.reset_index().rename(columns={"index": "ticker"})
StatChart(theme="dark", height=420, title="Days Held vs Return Contribution (bubble = avg weight)").scatter(
    scatter_df,
    x_col="days_held",
    y_col="contrib_bps",
    label_col="ticker",
    size_col="avg_weight_pct",
    color="#26a69a",
).show()


TypeError: StatChart.scatter() got an unexpected keyword argument 'x_col'

In [ ]:
t = trades.copy()
t["pnl_bps"] = t.get("pnl_bps", t.get("pnl", 0.0))
t["datetime"] = pd.to_datetime(t["datetime"]).dt.normalize()
t["days_held"] = t.get("days_held", np.nan)

by_ticker = (
    t.groupby("instrument")
     .agg(
         n_trades   =("pnl_bps","count"),
         total_pnl  =("pnl_bps","sum"),
         avg_pnl    =("pnl_bps","mean"),
         win_rate   =("pnl_bps", lambda x: (x > 0).mean()),
     )
     .sort_values("total_pnl", ascending=False)
)

display(by_ticker.head(15).style
    .format({"total_pnl":"{:.0f}","avg_pnl":"{:.1f}","win_rate":"{:.0%}"})
    .background_gradient(subset=["total_pnl"], cmap="Greens")
    .bar(subset=["win_rate"], color="#26a69a", vmin=0, vmax=1)
    .set_caption("Trade P&L by Ticker — Top 15")
)
display(by_ticker.tail(10).style
    .format({"total_pnl":"{:.0f}","avg_pnl":"{:.1f}","win_rate":"{:.0%}"})
    .background_gradient(subset=["total_pnl"], cmap="Reds_r")
    .bar(subset=["win_rate"], color="#ef5350", vmin=0, vmax=1)
    .set_caption("Trade P&L by Ticker — Bottom 10")
)

display(t.sort_values("datetime", ascending=False).head(10)
    .style.set_caption("Most Recent 10 Trades")
    .hide(axis="index"))
